# Sentiment-Driven Nifty Volatility Prediction

Predicts next-day realized (Garman-Klass) volatility of the Nifty 50 index
from sentiment features extracted from Indian financial news, validated
with **walk-forward** backtesting against naive persistence and GARCH(1,1)
baselines.

**Framing:** for each trading day, aggregate all news published between the
previous close (3:30 PM IST) and today's market open (9:15 AM IST) into a
feature vector, and predict that day's realized volatility with LightGBM.

**Before running:** in the notebook's right-hand Settings panel, turn on
**Internet** (required for `yfinance`, GDELT, RSS, and downloading FinBERT)
and set the **Accelerator** to **GPU T4 x2** or **GPU T4 x1** (FinBERT
inference is much faster on GPU, though it will fall back to CPU
automatically if none is available). Then **Run All**.

## Pipeline

```
News (GDELT historical + live RSS)        Price data (yfinance, ^NSEI OHLC)
            |                                       |
            v                                       |
Preprocess & sentiment (filter, dedupe, FinBERT)     |
            |                                       |
            +-------------------+-------------------+
                                v
                   Daily feature table
              (sentiment features + price/vol lags)
                                |
                                v
                  LightGBM (Optuna-tuned)
                                |
                                v
         Walk-forward validation & backtest
      (vs. naive persistence + GARCH, Sharpe/drawdown)
```


## 0. Setup

In [ ]:
!pip install -q feedparser optuna shap arch lightgbm yfinance --upgrade


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import time
import pathlib
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytz
import requests
import feedparser
import torch
import lightgbm as lgb
import optuna
import shap
from dateutil.relativedelta import relativedelta
from arch import arch_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import yfinance as yf

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (11, 4.5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# --- config (src/config.py) ---
"""Shared configuration constants for the Nifty volatility prediction pipeline."""

import pytz

TICKER = "^NSEI"
PRICE_LOOKBACK_YEARS = 5

IST = pytz.timezone("Asia/Kolkata")
PREV_CLOSE_TIME = "15:30"
TODAY_OPEN_TIME = "09:15"

# Keywords used both for GDELT queries and the local relevance filter.
RELEVANCE_KEYWORDS = [
    "nifty", "sensex", "rbi", "reserve bank of india", "budget", "inflation",
    "fii", "dii", "foreign institutional investor", "domestic institutional investor",
    "sebi", "repo rate", "gdp india", "indian rupee", "bse", "nse india",
    "reliance industries", "hdfc bank", "icici bank", "infosys", "tcs",
    "tata consultancy", "kotak mahindra", "larsen", "itc limited", "axis bank",
    "state bank of india", "bharti airtel", "bajaj finance", "hindustan unilever",
    "maruti suzuki", "sun pharma", "asian paints", "adani",
]

GDELT_QUERY = "(Nifty OR Sensex OR \"RBI\" OR \"Union Budget\" OR \"Indian rupee\" OR FII OR DII)"
GDELT_SOURCE_COUNTRY = "India"

RSS_FEEDS = {
    "moneycontrol": "https://www.moneycontrol.com/rss/marketreports.xml",
    "economic_times": "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
    "livemint": "https://www.livemint.com/rss/markets",
    "business_standard": "https://www.business-standard.com/rss/markets-106.rss",
    "google_news_nifty": "https://news.google.com/rss/search?q=Nifty+when:2d&hl=en-IN&gl=IN&ceid=IN:en",
}

FINBERT_MODEL_NAME = "ProsusAI/finbert"

RANDOM_SEED = 42


In [ ]:
# News backfill matches the price lookback (PRICE_LOOKBACK_YEARS, see config
# cell above) so sentiment features are available across the full training
# history -- otherwise the years without matching news just fall back to
# price-only features, which undercuts the whole point of this project.
# GDELT backfill walks this range in weekly chunks (~1 request/chunk), so
# this takes longer to collect (~10-15 min for 5 years) than a shorter
# window would; shrink PRICE_LOOKBACK_YEARS in the config cell for a faster
# smoke-test run.
GDELT_START_DATE = (datetime.utcnow() - timedelta(days=PRICE_LOOKBACK_YEARS * 365)).strftime("%Y-%m-%d")
GDELT_END_DATE = datetime.utcnow().strftime("%Y-%m-%d")


## 1. Price data

5+ years of Nifty 50 daily OHLC from `yfinance` (free, no key, avoids
fighting NSE's own bot protection). We derive the **Garman-Klass**
volatility target here too, since it only needs OHLC:

```
sigma^2_GK = 0.5 * (ln(H/L))^2 - (2*ln2 - 1) * (ln(C/O))^2
```

more informative than a plain `std(returns)` estimate.

In [ ]:
"""Price data collection and Garman-Klass volatility target construction."""

import numpy as np
import pandas as pd
import yfinance as yf


def fetch_price_data(ticker: str, years: int = 5) -> pd.DataFrame:
    """Download daily OHLC data for `ticker` from yfinance.

    Returns a DataFrame indexed by date with columns:
    open, high, low, close, volume
    """
    df = yf.download(ticker, period=f"{years}y", interval="1d", auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)
    df = df[["open", "high", "low", "close", "volume"]].dropna()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "date"
    return df


def garman_klass_volatility(df: pd.DataFrame) -> pd.Series:
    """Garman-Klass variance estimator for each row's own OHLC, annualization-free.

    sigma^2_GK = 0.5*(ln(H/L))^2 - (2*ln2 - 1)*(ln(C/O))^2

    Returns the volatility (sqrt of the variance estimate), floored at 0 to
    avoid sqrt of small negative numbers caused by numerical noise.
    """
    log_hl = np.log(df["high"] / df["low"])
    log_co = np.log(df["close"] / df["open"])
    gk_var = 0.5 * log_hl ** 2 - (2 * np.log(2) - 1) * log_co ** 2
    gk_var = gk_var.clip(lower=0)
    return np.sqrt(gk_var)


def build_price_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add return, Garman-Klass vol, rolling vol, and day-of-week features.

    `target_next_gk_vol` is the Garman-Klass volatility realized on the
    *next* trading day — this is the regression target. Everything else is
    computed from information available as of the close of `date`, so it is
    safe to use as a same-day feature.
    """
    out = df.copy()
    out["log_return"] = np.log(out["close"] / out["close"].shift(1))
    # gk_vol at row `date` is computed from that day's own OHLC, so it is
    # only known at that day's close -- it is a valid *same-day* feature
    # for predicting the *next* day's volatility (i.e. "yesterday's vol"
    # relative to the day being forecast).
    out["gk_vol"] = garman_klass_volatility(out)
    out["gk_vol_lag1"] = out["gk_vol"].shift(1)
    out["gk_vol_ma5"] = out["gk_vol"].rolling(5).mean()
    out["gk_vol_ma10"] = out["gk_vol"].rolling(10).mean()
    out["day_of_week"] = out.index.dayofweek

    # Target: next day's realized Garman-Klass volatility.
    out["target_next_gk_vol"] = out["gk_vol"].shift(-1)
    out["target_next_log_gk_vol"] = np.log(out["target_next_gk_vol"] + 1e-8)

    return out


In [ ]:
prices_raw = fetch_price_data(TICKER, PRICE_LOOKBACK_YEARS)
price_features = build_price_features(prices_raw)
print(price_features.shape)
price_features.tail()


## 2. News collection

- **GDELT DOC 2.0 API** for historical backfill (headline + timestamp +
  source; its own tone score is aggregate-only, so every headline --
  GDELT and RSS alike -- gets scored by FinBERT below for one consistent
  signal).
- **Live RSS** (Moneycontrol, Economic Times, LiveMint, Business Standard,
  Google News) for ongoing collection. Re-running this cell on a schedule
  (e.g. a daily Kaggle job) extends the dataset going forward and doubles
  as a genuine out-of-sample test on news collected *after* the model was
  built.

In [ ]:
"""Historical news backfill via the GDELT 2.0 DOC API.

We use the DOC API (article search) rather than raw GKG export files: GKG
files are large daily dumps meant for bulk download, while the DOC API lets
us search directly for Nifty/Sensex/RBI-relevant coverage and get back
headline + timestamp + source, which is exactly what the feature pipeline
needs. Its own tone score is aggregate-only (not per article) so we skip it
here and score every headline -- GDELT and RSS alike -- with FinBERT for a
single, consistent sentiment signal (see src/sentiment/finbert.py).
"""

import time
from datetime import datetime, timedelta

import pandas as pd
import requests

GDELT_DOC_ENDPOINT = "https://api.gdeltproject.org/api/v2/doc/doc"


def _query_gdelt_window(query: str, start: datetime, end: datetime, max_records: int = 250,
                         source_country: str | None = None, timeout: int = 30) -> list[dict]:
    """Query GDELT DOC API for one time window. Returns a list of article dicts."""
    full_query = query
    if source_country:
        full_query = f"{query} sourcecountry:{source_country}"

    params = {
        "query": full_query,
        "mode": "artlist",
        "maxrecords": max_records,
        "format": "json",
        "startdatetime": start.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end.strftime("%Y%m%d%H%M%S"),
        "sort": "datedesc",
    }
    try:
        resp = requests.get(GDELT_DOC_ENDPOINT, params=params, timeout=timeout,
                             headers={"User-Agent": "Mozilla/5.0 (research; nifty-vol-pipeline)"})
        resp.raise_for_status()
        data = resp.json()
    except (requests.RequestException, ValueError):
        return []
    return data.get("articles", [])


def collect_gdelt_history(query: str, start_date: str, end_date: str,
                           source_country: str | None = "India",
                           chunk_days: int = 7, sleep_sec: float = 1.0,
                           max_records: int = 250) -> pd.DataFrame:
    """Backfill historical news by walking `start_date`..`end_date` in
    `chunk_days`-sized windows, one GDELT DOC API request per window.

    Returns a DataFrame with columns: headline, source, published_at, url
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    rows = []
    cursor = start
    while cursor < end:
        window_end = min(cursor + timedelta(days=chunk_days), end)
        articles = _query_gdelt_window(query, cursor, window_end, max_records=max_records,
                                        source_country=source_country)
        for art in articles:
            seendate = art.get("seendate")
            try:
                published_at = datetime.strptime(seendate, "%Y%m%dT%H%M%SZ")
            except (TypeError, ValueError):
                continue
            rows.append({
                "headline": art.get("title", "").strip(),
                "source": art.get("domain", "gdelt"),
                "published_at": published_at,
                "url": art.get("url", ""),
            })
        cursor = window_end
        time.sleep(sleep_sec)

    df = pd.DataFrame(rows, columns=["headline", "source", "published_at", "url"])
    df = df.dropna(subset=["headline"])
    df = df[df["headline"].str.len() > 0]
    return df.drop_duplicates(subset=["url"]).reset_index(drop=True)


In [ ]:
"""Live RSS collection from Indian financial news outlets + Google News.

Meant to be run on a schedule (e.g. a daily Kaggle/cron job) so that it
extends the GDELT-backed historical dataset going forward, and doubles as a
genuine out-of-sample test on news collected *after* the model was built.
"""

from datetime import datetime

import feedparser
import pandas as pd


def _parse_entry_time(entry) -> datetime | None:
    for key in ("published_parsed", "updated_parsed"):
        t = getattr(entry, key, None)
        if t is not None:
            return datetime(*t[:6])
    return None


def fetch_rss_feed(name: str, url: str, timeout: int = 15) -> pd.DataFrame:
    """Fetch and parse a single RSS feed into headline rows."""
    parsed = feedparser.parse(url, request_headers={"User-Agent": "Mozilla/5.0"})
    rows = []
    for entry in parsed.entries:
        headline = getattr(entry, "title", "").strip()
        if not headline:
            continue
        published_at = _parse_entry_time(entry) or datetime.utcnow()
        rows.append({
            "headline": headline,
            "source": name,
            "published_at": published_at,
            "url": getattr(entry, "link", ""),
        })
    return pd.DataFrame(rows, columns=["headline", "source", "published_at", "url"])


def collect_rss_news(feeds: dict) -> pd.DataFrame:
    """Fetch all configured RSS feeds and return a combined, deduped frame."""
    frames = [fetch_rss_feed(name, url) for name, url in feeds.items()]
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame(columns=["headline", "source", "published_at", "url"])
    df = pd.concat(frames, ignore_index=True)
    return df.drop_duplicates(subset=["url"]).reset_index(drop=True)


In [ ]:
gdelt_news = collect_gdelt_history(
    query=GDELT_QUERY,
    start_date=GDELT_START_DATE,
    end_date=GDELT_END_DATE,
    source_country=GDELT_SOURCE_COUNTRY,
    chunk_days=7,
)
print(f"GDELT: {len(gdelt_news)} articles")

rss_news = collect_rss_news(RSS_FEEDS)
print(f"RSS (live, today): {len(rss_news)} articles")

raw_news = pd.concat([gdelt_news, rss_news], ignore_index=True).drop_duplicates(subset=["url"])
print(f"Combined: {len(raw_news)} articles")
raw_news.tail()


## 3. Preprocessing

- **Dedupe** near-identical headlines (wire content gets republished
  across outlets; without deduping, "article count" mostly measures
  republishing, not distinct news).
- **Relevance filter** on Nifty/Sensex/RBI/budget/inflation/FII-DII/top
  constituent names.
- **Trading-day bucketing**: news published between previous close
  (3:30 PM IST) and today's open (9:15 AM IST) is assigned to *today*.

In [ ]:
"""Near-duplicate headline removal.

Wire content gets republished verbatim (or near-verbatim) across outlets, so
naive article counts mostly measure republishing rather than distinct news.
We normalize text and drop exact normalized duplicates cheaply, then run a
fuzzy pass with difflib *within each same-day bucket only* -- comparing
every headline in the whole corpus pairwise would be O(n^2) and is not
needed since near-duplicates of the same story appear on the same day.
"""

import re
from difflib import SequenceMatcher

import pandas as pd

_WS_PUNCT_RE = re.compile(r"[^a-z0-9\s]")


def normalize_headline(text: str) -> str:
    text = text.lower().strip()
    text = _WS_PUNCT_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def dedupe_headlines(df: pd.DataFrame, fuzzy_threshold: float = 0.9) -> pd.DataFrame:
    """Drop exact and near-duplicate headlines.

    Expects a `headline` column and a `published_at` (datetime) column.
    """
    if df.empty:
        return df

    out = df.copy()
    out["_norm"] = out["headline"].map(normalize_headline)
    out = out.drop_duplicates(subset=["_norm"])

    out["_date"] = pd.to_datetime(out["published_at"]).dt.date
    keep_mask = pd.Series(True, index=out.index)

    for _, group in out.groupby("_date"):
        seen: list[tuple] = []  # (index, normalized text)
        for idx, norm in group["_norm"].items():
            is_dup = False
            for _, seen_norm in seen:
                if SequenceMatcher(None, norm, seen_norm).ratio() >= fuzzy_threshold:
                    is_dup = True
                    break
            if is_dup:
                keep_mask.loc[idx] = False
            else:
                seen.append((idx, norm))

    out = out[keep_mask].drop(columns=["_norm", "_date"])
    return out.reset_index(drop=True)


In [ ]:
"""Keyword relevance filtering.

GDELT and Google News RSS return a lot of noise for broad market queries,
so we keep only headlines that mention Nifty/Sensex/macro terms or a top
Nifty50 constituent name.
"""

import re

import pandas as pd


def _compile_keyword_pattern(keywords: list[str]) -> re.Pattern:
    escaped = [re.escape(k.lower()) for k in keywords]
    return re.compile(r"\b(" + "|".join(escaped) + r")\b")


def filter_relevant(df: pd.DataFrame, keywords: list[str]) -> pd.DataFrame:
    """Keep only rows whose `headline` contains one of `keywords` (case-insensitive)."""
    if df.empty:
        return df
    pattern = _compile_keyword_pattern(keywords)
    mask = df["headline"].str.lower().apply(lambda h: bool(pattern.search(h)))
    return df[mask].reset_index(drop=True)


In [ ]:
"""Trading-day bucketing.

A news item published at time `ts` (assumed UTC, the norm for GDELT
`seendate` and feedparser's `*_parsed` fields) is assigned to the trading
day `D` such that `ts` falls in the window `(D-1's 15:30 IST close, D's
09:15 IST open]`.

Rather than explicitly computing "previous close", we use an equivalent
and simpler rule: `D` is the *earliest* trading day whose 09:15 IST open is
at or after `ts`. This is exactly the same window (news during trading
hours or after a trading day's close is deferred to the *next* trading
day's pre-market window), and it is robust to weekends/holidays because it
only ever matches against real trading days pulled from the price index.
"""

import numpy as np
import pandas as pd
import pytz

IST = pytz.timezone("Asia/Kolkata")
OPEN_HOUR, OPEN_MINUTE = 9, 15


def _trading_day_opens(trading_days: pd.DatetimeIndex) -> pd.DatetimeIndex:
    days = pd.DatetimeIndex(sorted(pd.to_datetime(trading_days)))
    opens = days + pd.Timedelta(hours=OPEN_HOUR, minutes=OPEN_MINUTE)
    return opens.tz_localize(IST)


def bucket_news_to_trading_days(news_df: pd.DataFrame, trading_days: pd.DatetimeIndex) -> pd.DataFrame:
    """Assign each news row to a trading day (added as a `trading_day` column).

    Rows published after the open of the last available trading day (i.e.
    with no future trading day to bucket into yet) are dropped.
    """
    if news_df.empty:
        out = news_df.copy()
        out["trading_day"] = pd.Series(dtype="datetime64[ns]")
        return out

    out = news_df.copy()
    ts = pd.to_datetime(out["published_at"])
    if ts.dt.tz is None:
        ts = ts.dt.tz_localize("UTC")
    ts_ist = ts.dt.tz_convert(IST)

    opens = _trading_day_opens(trading_days)
    days = pd.DatetimeIndex(sorted(pd.to_datetime(trading_days)))

    positions = np.searchsorted(opens.values, ts_ist.values, side="left")
    valid = positions < len(days)

    out = out.loc[valid].copy()
    out["trading_day"] = days[positions[valid]]
    return out.reset_index(drop=True)


In [ ]:
news_deduped = dedupe_headlines(raw_news)
news_relevant = filter_relevant(news_deduped, RELEVANCE_KEYWORDS)
news_bucketed = bucket_news_to_trading_days(news_relevant, price_features.index)

print(f"raw={len(raw_news)}  deduped={len(news_deduped)}  relevant={len(news_relevant)}  bucketed={len(news_bucketed)}")
news_bucketed.tail()


## 4. Sentiment scoring (FinBERT)

`ProsusAI/finbert`, pretrained, inference only. Collapses the
positive/negative/neutral triple into a single polarity score:
`P(positive) - P(negative)`.

In [ ]:
"""FinBERT sentiment scoring for news headlines.

Uses ProsusAI/finbert (pretrained, inference only) to score each headline's
positive/negative/neutral probabilities, then collapses that triple into a
single polarity score: P(positive) - P(negative). Runs fine in batches on a
free Colab/Kaggle T4 GPU, and falls back to CPU automatically if no GPU is
available.
"""

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer



class FinBertScorer:
    def __init__(self, model_name: str = FINBERT_MODEL_NAME, device: str | None = None, batch_size: int = 32):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        # ProsusAI/finbert label order: 0=positive, 1=negative, 2=neutral
        self.id2label = self.model.config.id2label

    @torch.no_grad()
    def score_batch(self, headlines: list[str]) -> np.ndarray:
        inputs = self.tokenizer(headlines, padding=True, truncation=True, max_length=64,
                                 return_tensors="pt").to(self.device)
        logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        return probs

    def score_all(self, headlines: pd.Series) -> pd.DataFrame:
        headlines = headlines.fillna("").astype(str).tolist()
        label_order = [self.id2label[i].lower() for i in range(len(self.id2label))]
        pos_idx = label_order.index("positive")
        neg_idx = label_order.index("negative")

        all_probs = []
        for start in range(0, len(headlines), self.batch_size):
            batch = headlines[start:start + self.batch_size]
            if not batch:
                continue
            all_probs.append(self.score_batch(batch))
        probs = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, len(label_order)))

        result = pd.DataFrame(probs, columns=label_order)
        result["polarity"] = result["positive"] - result["negative"]
        return result


def score_headlines(df: pd.DataFrame, scorer: FinBertScorer | None = None, batch_size: int = 32) -> pd.DataFrame:
    """Score a news DataFrame's `headline` column, returning the frame with
    `positive`, `negative`, `neutral`, `polarity` columns appended."""
    scorer = scorer or FinBertScorer(batch_size=batch_size)
    scores = scorer.score_all(df["headline"])
    out = df.reset_index(drop=True).copy()
    for col in ("positive", "negative", "neutral", "polarity"):
        out[col] = scores[col].values
    return out


In [ ]:
finbert = FinBertScorer(batch_size=32)
scored_news = score_headlines(news_bucketed, scorer=finbert)
print(f"Device: {finbert.device}")
scored_news[["headline", "positive", "negative", "neutral", "polarity"]].tail()


## 5. Feature table

Sentiment: mean/std polarity, article count, %positive, %negative, plus
3-day and 5-day rolling means (news effects aren't always priced in
same-day). Price/autoregressive: yesterday's Garman-Klass vol, 5/10-day
average vol, yesterday's return, day-of-week (documented Monday effect in
Indian markets).

Volatility is highly autocorrelated, so naive persistence is already a
strong baseline -- the price lags are included as *features*, not just a
baseline to beat, so sentiment has to earn its place on top of them.

In [ ]:
"""Daily feature table: sentiment aggregates (same-day + rolling) joined to
price/autoregressive features.

Volatility is highly autocorrelated, so the naive baseline ("tomorrow's vol
~= today's vol") is already strong. The price-based lags are included here
as *features*, not just as a baseline to beat -- sentiment has to earn its
keep on top of them.
"""

import numpy as np
import pandas as pd

PRICE_FEATURE_COLS = [
    "gk_vol", "gk_vol_lag1", "gk_vol_ma5", "gk_vol_ma10", "log_return", "day_of_week",
]

SENTIMENT_BASE_COLS = [
    "sent_mean", "sent_std", "article_count", "pct_positive", "pct_negative",
]


def aggregate_daily_sentiment(scored_news: pd.DataFrame) -> pd.DataFrame:
    """Aggregate FinBERT-scored, trading-day-bucketed news into one row per
    trading day: mean/std polarity, article count, %positive, %negative."""
    if scored_news.empty:
        return pd.DataFrame(columns=["trading_day"] + SENTIMENT_BASE_COLS)

    news = scored_news.copy()
    news["is_positive"] = news["positive"] > news["negative"]
    news["is_negative"] = news["negative"] > news["positive"]

    grouped = news.groupby("trading_day")
    daily = grouped.agg(
        sent_mean=("polarity", "mean"),
        sent_std=("polarity", "std"),
        article_count=("polarity", "size"),
        pct_positive=("is_positive", "mean"),
        pct_negative=("is_negative", "mean"),
    ).reset_index()
    daily["sent_std"] = daily["sent_std"].fillna(0.0)
    return daily


def add_rolling_sentiment(daily_sentiment: pd.DataFrame) -> pd.DataFrame:
    """Add 3-day and 5-day rolling means of the sentiment aggregates --
    news effects aren't always priced in same-day."""
    out = daily_sentiment.sort_values("trading_day").reset_index(drop=True)
    for col in SENTIMENT_BASE_COLS:
        out[f"{col}_roll3"] = out[col].rolling(3, min_periods=1).mean()
        out[f"{col}_roll5"] = out[col].rolling(5, min_periods=1).mean()
    return out


def build_feature_table(price_features: pd.DataFrame, scored_news: pd.DataFrame) -> pd.DataFrame:
    """Join sentiment features to price features on trading day.

    `price_features` is expected to already have `target_next_gk_vol` /
    `target_next_log_gk_vol` columns (see src/data/price_data.py).
    """
    daily_sentiment = aggregate_daily_sentiment(scored_news)
    daily_sentiment = add_rolling_sentiment(daily_sentiment)

    prices = price_features.reset_index().rename(columns={"date": "trading_day"})
    merged = prices.merge(daily_sentiment, on="trading_day", how="left")

    sentiment_cols = [c for c in merged.columns
                       if c.startswith(tuple(SENTIMENT_BASE_COLS))]
    for col in sentiment_cols:
        if col == "article_count" or col.startswith("article_count"):
            merged[col] = merged[col].fillna(0)
        else:
            merged[col] = merged[col].fillna(0.0)

    merged["has_news"] = (merged["article_count"] > 0).astype(int)
    merged = merged.set_index("trading_day")
    merged.index.name = "date"
    return merged


def get_feature_columns(feature_table: pd.DataFrame) -> list[str]:
    exclude = {"open", "high", "low", "close", "volume", "target_next_gk_vol", "target_next_log_gk_vol"}
    return [c for c in feature_table.columns if c not in exclude]


In [ ]:
feature_table = build_feature_table(price_features, scored_news)
feature_cols = get_feature_columns(feature_table)
print(f"{len(feature_cols)} features, {len(feature_table)} rows")
print(feature_cols)
feature_table[feature_cols + ["target_next_gk_vol"]].tail()


## 6. Baselines: naive persistence + GARCH(1,1)

Reported on **log-volatility** (right-skewed, strictly positive target).
These are the bars the LightGBM model has to clear.

In [ ]:
"""Baselines the LightGBM model has to beat: naive persistence and GARCH(1,1)."""

import numpy as np
import pandas as pd
from arch import arch_model


def naive_persistence_forecast(gk_vol_today: pd.Series) -> pd.Series:
    """Tomorrow's vol forecast = today's realized vol."""
    return gk_vol_today


def fit_garch_forecast(returns_pct: pd.Series, refit_every: int = 1) -> pd.Series:
    """Rolling one-step-ahead GARCH(1,1) volatility forecast.

    `returns_pct` should be percentage log returns (e.g. 100 * log_return)
    -- `arch` is numerically happier with returns scaled to roughly O(1).
    Refits the model at every step by default (set `refit_every` > 1 to
    refit less often and reuse the last fitted params in between, which is
    much faster over long walk-forward windows).

    Returns a Series of one-step-ahead conditional volatility forecasts,
    indexed like `returns_pct`: the value at date d is a forecast of date
    d's own return volatility, made using only history strictly before d.
    To compare against `target_next_gk_vol` (indexed by the *feature* date
    T, holding the forecast for T+1) the caller must shift this series by
    -1, e.g. `fit_garch_forecast(...).shift(-1)`.
    """
    returns_pct = returns_pct.dropna()
    forecasts = pd.Series(index=returns_pct.index, dtype=float)

    last_params = None
    for i in range(1, len(returns_pct)):
        history = returns_pct.iloc[:i]
        if len(history) < 30:
            continue
        try:
            if last_params is None or i % refit_every == 0:
                model = arch_model(history, vol="GARCH", p=1, q=1, mean="Zero", rescale=False)
                res = model.fit(disp="off")
                last_params = res.params
            else:
                model = arch_model(history, vol="GARCH", p=1, q=1, mean="Zero", rescale=False)
                res = model.fix(last_params)
            fcast = res.forecast(horizon=1, reindex=False)
            vol_forecast = np.sqrt(fcast.variance.values[-1, 0]) / 100.0
        except Exception:
            vol_forecast = np.nan
        forecasts.iloc[i] = vol_forecast

    return forecasts


In [ ]:
returns_pct_full = (feature_table["log_return"] * 100).dropna()
garch_vol_forecast = fit_garch_forecast(returns_pct_full, refit_every=5)
# fit_garch_forecast(...) at date d forecasts d's own vol from history
# before d; shift(-1) realigns it to the *feature* date (T -> forecast
# for T+1), matching target_next_log_gk_vol and the naive/model forecasts.
garch_log_vol_forecast = np.log(garch_vol_forecast + 1e-8).shift(-1)
garch_log_vol_forecast.tail()


## 7. LightGBM model + Optuna tuning

In [ ]:
"""LightGBM volatility model with Optuna hyperparameter tuning.

Trained on log-volatility (`target_next_log_gk_vol`) since realized
volatility is right-skewed and strictly positive -- see src/validation for
where RMSE/MAE are reported on this same log scale.
"""

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd


optuna.logging.set_verbosity(optuna.logging.WARNING)

DEFAULT_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "seed": RANDOM_SEED,
}


def _objective(trial, X_train, y_train, X_val, y_val):
    params = {
        **DEFAULT_PARAMS,
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    preds = model.predict(X_val, num_iteration=model.best_iteration_)
    return float(np.sqrt(np.mean((preds - y_val) ** 2)))


def tune_hyperparameters(X_train, y_train, X_val, y_val, n_trials: int = 30) -> dict:
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(lambda t: _objective(t, X_train, y_train, X_val, y_val), n_trials=n_trials, show_progress_bar=False)
    return {**DEFAULT_PARAMS, **study.best_params}


def train_lgbm(X_train, y_train, X_val=None, y_val=None, params: dict | None = None,
                objective: str = "regression", alpha: float | None = None) -> lgb.LGBMRegressor:
    """Train a single LightGBM regressor.

    Set `objective="quantile"` with `alpha` in {0.1, 0.5, 0.9} to fit one
    quantile of the predictive distribution -- cheap uncertainty bands for
    gradient boosting, unlike for a deep net.
    """
    fit_params = {**DEFAULT_PARAMS, **(params or {})}
    fit_params["objective"] = objective
    if objective == "quantile":
        fit_params["alpha"] = alpha
    model = lgb.LGBMRegressor(**fit_params)
    if X_val is not None:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])
    else:
        model.fit(X_train, y_train)
    return model


def train_quantile_ensemble(X_train, y_train, X_val, y_val, params: dict,
                             quantiles=(0.1, 0.5, 0.9)) -> dict:
    """Train one LightGBM model per quantile, sharing tuned point-forecast params."""
    models = {}
    for q in quantiles:
        models[q] = train_lgbm(X_train, y_train, X_val, y_val, params=params,
                                objective="quantile", alpha=q)
    return models


## 8. Walk-forward validation

Train on an expanding ~1yr window, test on the next ~2 months, slide
forward, repeat -- never a random shuffle split. Headline result: the %
improvement in RMSE over naive persistence and GARCH(1,1), not a
standalone error number.

In [ ]:
"""Walk-forward validation.

Train on an expanding window of ~1 year, test on the next ~2 months, slide
forward and repeat. Never a random shuffle split -- that's the single most
common way this kind of project quietly leaks future information into
training. Errors are reported on log-volatility (right-skewed, strictly
positive target), and the headline result is the margin by which the model
beats naive persistence and GARCH(1,1), not a standalone error number.
"""

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta



@dataclass
class FoldResult:
    fold: int
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp
    n_train: int
    n_test: int
    rmse_model: float
    mae_model: float
    rmse_naive: float
    mae_naive: float
    rmse_garch: float
    mae_garch: float
    predictions: pd.DataFrame = field(repr=False)


def add_naive_forecast_column(feature_table: pd.DataFrame, col_name: str = "naive_log_vol") -> pd.DataFrame:
    """Naive persistence forecast for tomorrow's log-vol = log(today's realized GK vol)."""
    out = feature_table.copy()
    out[col_name] = np.log(out["gk_vol"] + 1e-8)
    return out


def generate_walk_forward_folds(dates: pd.DatetimeIndex, train_years: int = 1,
                                 test_months: int = 2, step_months: int = 2):
    """Yield (train_mask, test_mask) boolean arrays over `dates`."""
    dates = pd.DatetimeIndex(dates)
    start = dates.min()
    train_end = start + relativedelta(years=train_years)

    while True:
        test_end = train_end + relativedelta(months=test_months)
        train_mask = (dates >= start) & (dates < train_end)
        test_mask = (dates >= train_end) & (dates < test_end)
        if test_mask.sum() == 0:
            break
        yield train_mask, test_mask
        train_end = train_end + relativedelta(months=step_months)
        if train_end >= dates.max():
            break


def _rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def _mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))


def run_walk_forward(feature_table: pd.DataFrame, feature_cols: list[str],
                      naive_log_vol_col: str, garch_log_vol: pd.Series,
                      train_years: int = 1, test_months: int = 2, step_months: int = 2,
                      n_optuna_trials: int = 20) -> list[FoldResult]:
    """Run the full walk-forward loop, tuning + training a fresh LightGBM
    model on each expanding training window and comparing against naive
    persistence and GARCH(1,1) on the held-out test window.

    `garch_log_vol` must already be indexed like `feature_table` (i.e. the
    value at feature date T is the forecast for T+1's log-vol) -- see
    `fit_garch_forecast`'s docstring for the `.shift(-1)` needed to get
    there from its raw output.
    """
    df = feature_table.dropna(subset=["target_next_log_gk_vol"]).copy()
    dates = df.index

    results = []
    for fold_id, (train_mask, test_mask) in enumerate(
            generate_walk_forward_folds(dates, train_years, test_months, step_months)):

        train_df = df.loc[train_mask]
        test_df = df.loc[test_mask]
        if len(train_df) < 60 or len(test_df) == 0:
            continue

        # Hold out the tail of the training window for early stopping / Optuna validation.
        val_cut = int(len(train_df) * 0.85)
        X_tr, y_tr = train_df[feature_cols].iloc[:val_cut], train_df["target_next_log_gk_vol"].iloc[:val_cut]
        X_val, y_val = train_df[feature_cols].iloc[val_cut:], train_df["target_next_log_gk_vol"].iloc[val_cut:]
        X_test, y_test = test_df[feature_cols], test_df["target_next_log_gk_vol"]

        best_params = tune_hyperparameters(X_tr, y_tr, X_val, y_val, n_trials=n_optuna_trials)
        model = train_lgbm(train_df[feature_cols], train_df["target_next_log_gk_vol"], params=best_params)
        preds = model.predict(X_test)

        naive_preds = test_df[naive_log_vol_col].values
        garch_preds = garch_log_vol.reindex(test_df.index).values
        garch_valid = ~np.isnan(garch_preds)

        pred_frame = pd.DataFrame({
            "date": test_df.index,
            "y_true_log": y_test.values,
            "y_pred_model_log": preds,
            "y_pred_naive_log": naive_preds,
            "y_pred_garch_log": garch_preds,
        })

        results.append(FoldResult(
            fold=fold_id,
            train_start=train_df.index.min(), train_end=train_df.index.max(),
            test_start=test_df.index.min(), test_end=test_df.index.max(),
            n_train=len(train_df), n_test=len(test_df),
            rmse_model=_rmse(y_test, preds), mae_model=_mae(y_test, preds),
            rmse_naive=_rmse(y_test, naive_preds), mae_naive=_mae(y_test, naive_preds),
            rmse_garch=_rmse(y_test.values[garch_valid], garch_preds[garch_valid]) if garch_valid.any() else np.nan,
            mae_garch=_mae(y_test.values[garch_valid], garch_preds[garch_valid]) if garch_valid.any() else np.nan,
            predictions=pred_frame,
        ))

    return results


def summarize_folds(results: list[FoldResult]) -> pd.DataFrame:
    rows = [{
        "fold": r.fold, "test_start": r.test_start.date(), "test_end": r.test_end.date(),
        "n_test": r.n_test,
        "rmse_model": r.rmse_model, "rmse_naive": r.rmse_naive, "rmse_garch": r.rmse_garch,
        "mae_model": r.mae_model, "mae_naive": r.mae_naive, "mae_garch": r.mae_garch,
        "rmse_improvement_vs_naive_pct": 100 * (r.rmse_naive - r.rmse_model) / r.rmse_naive,
        "rmse_improvement_vs_garch_pct": 100 * (r.rmse_garch - r.rmse_model) / r.rmse_garch if not np.isnan(r.rmse_garch) else np.nan,
    } for r in results]
    return pd.DataFrame(rows)


def overall_metrics(results: list[FoldResult]) -> dict:
    all_preds = pd.concat([r.predictions for r in results], ignore_index=True)
    garch_valid = all_preds["y_pred_garch_log"].notna()
    return {
        "rmse_model": _rmse(all_preds["y_true_log"], all_preds["y_pred_model_log"]),
        "mae_model": _mae(all_preds["y_true_log"], all_preds["y_pred_model_log"]),
        "rmse_naive": _rmse(all_preds["y_true_log"], all_preds["y_pred_naive_log"]),
        "mae_naive": _mae(all_preds["y_true_log"], all_preds["y_pred_naive_log"]),
        "rmse_garch": _rmse(all_preds.loc[garch_valid, "y_true_log"], all_preds.loc[garch_valid, "y_pred_garch_log"]),
        "mae_garch": _mae(all_preds.loc[garch_valid, "y_true_log"], all_preds.loc[garch_valid, "y_pred_garch_log"]),
        "n_predictions": len(all_preds),
    }


In [ ]:
feature_table_wf = add_naive_forecast_column(feature_table)

wf_results = run_walk_forward(
    feature_table_wf, feature_cols,
    naive_log_vol_col="naive_log_vol",
    garch_log_vol=garch_log_vol_forecast,
    train_years=1, test_months=2, step_months=2,
    n_optuna_trials=20,
)

fold_summary = summarize_folds(wf_results)
overall = overall_metrics(wf_results)

print(f"Ran {len(wf_results)} walk-forward folds\n")
display(fold_summary)
print("\nOverall (all folds pooled):")
for k, v in overall.items():
    print(f"  {k}: {v:.5f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\nModel beats naive persistence by "
      f"{100*(overall['rmse_naive']-overall['rmse_model'])/overall['rmse_naive']:.1f}% RMSE")
print(f"Model beats GARCH(1,1) by "
      f"{100*(overall['rmse_garch']-overall['rmse_model'])/overall['rmse_garch']:.1f}% RMSE")


In [ ]:
fig, ax = plt.subplots()
all_preds = pd.concat([r.predictions for r in wf_results], ignore_index=True).set_index("date").sort_index()
ax.plot(all_preds.index, np.exp(all_preds["y_true_log"]), label="Realized GK vol", lw=1.5)
ax.plot(all_preds.index, np.exp(all_preds["y_pred_model_log"]), label="LightGBM forecast", lw=1)
ax.plot(all_preds.index, np.exp(all_preds["y_pred_naive_log"]), label="Naive persistence", lw=0.8, alpha=0.6)
ax.set_title("Walk-forward: predicted vs. realized next-day volatility (all test folds)")
ax.legend()
plt.show()


## 9. Final model (most recent window) + quantile bands

Refit on the most recent training window for the backtest and SHAP
sections below, plus a cheap 10/50/90th-percentile quantile ensemble for
an uncertainty band.

In [ ]:
final_train = feature_table_wf.dropna(subset=["target_next_log_gk_vol"])
val_cut = int(len(final_train) * 0.85)
X_tr, y_tr = final_train[feature_cols].iloc[:val_cut], final_train["target_next_log_gk_vol"].iloc[:val_cut]
X_val, y_val = final_train[feature_cols].iloc[val_cut:], final_train["target_next_log_gk_vol"].iloc[val_cut:]

final_params = tune_hyperparameters(X_tr, y_tr, X_val, y_val, n_trials=30)
final_model = train_lgbm(final_train[feature_cols], final_train["target_next_log_gk_vol"], params=final_params)

quantile_models = train_quantile_ensemble(X_tr, y_tr, X_val, y_val, final_params)
print("Final params:", final_params)


## 10. Backtest

Simple volatility-targeting rule: position size scaled inversely to
predicted volatility, capped, with a 7.5 bps transaction cost per unit of
position change. Reports Sharpe ratio and max drawdown vs. buy-and-hold.

In [ ]:
"""Volatility-targeting backtest.

Position size is scaled inversely to predicted volatility: a larger
position when the model predicts low volatility, a smaller one when it
predicts high volatility, capped at `max_leverage`. Transaction costs are
charged on every change in position -- a backtest without costs is not a
credible backtest.
"""

import numpy as np
import pandas as pd

TRADING_DAYS_PER_YEAR = 252


def vol_target_positions(predicted_vol: pd.Series, target_daily_vol: float = 0.01,
                          max_leverage: float = 2.0) -> pd.Series:
    """position_t = clip(target_daily_vol / predicted_vol_t, 0, max_leverage)."""
    raw = target_daily_vol / predicted_vol.replace(0, np.nan)
    return raw.clip(lower=0, upper=max_leverage).fillna(0.0)


def run_backtest(predicted_vol: pd.Series, realized_return: pd.Series,
                  target_daily_vol: float = 0.01, max_leverage: float = 2.0,
                  cost_bps: float = 7.5) -> pd.DataFrame:
    """Run the vol-targeting strategy and a buy-and-hold benchmark side by side.

    `predicted_vol` and `realized_return` must be aligned on the same date
    index (`realized_return` is the *next* trading day's actual log return,
    i.e. the return realized while holding the position sized off
    `predicted_vol`).
    """
    idx = predicted_vol.index.intersection(realized_return.index)
    pred_vol = predicted_vol.loc[idx]
    ret = realized_return.loc[idx]

    positions = vol_target_positions(pred_vol, target_daily_vol, max_leverage)
    position_change = positions.diff().abs().fillna(positions.abs())
    cost = position_change * (cost_bps / 10_000)

    strategy_return = positions * ret - cost
    buy_hold_return = ret

    out = pd.DataFrame({
        "position": positions,
        "realized_return": ret,
        "strategy_return": strategy_return,
        "buy_hold_return": buy_hold_return,
        "transaction_cost": cost,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    out["buy_hold_equity"] = (1 + out["buy_hold_return"]).cumprod()
    return out


def sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.0) -> float:
    excess = returns - risk_free_rate / TRADING_DAYS_PER_YEAR
    if excess.std() == 0 or excess.empty:
        return 0.0
    return float(np.sqrt(TRADING_DAYS_PER_YEAR) * excess.mean() / excess.std())


def max_drawdown(equity_curve: pd.Series) -> float:
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1.0
    return float(drawdown.min())


def backtest_summary(backtest_df: pd.DataFrame) -> dict:
    return {
        "strategy_sharpe": sharpe_ratio(backtest_df["strategy_return"]),
        "buy_hold_sharpe": sharpe_ratio(backtest_df["buy_hold_return"]),
        "strategy_max_drawdown": max_drawdown(backtest_df["strategy_equity"]),
        "buy_hold_max_drawdown": max_drawdown(backtest_df["buy_hold_equity"]),
        "strategy_total_return": float(backtest_df["strategy_equity"].iloc[-1] - 1),
        "buy_hold_total_return": float(backtest_df["buy_hold_equity"].iloc[-1] - 1),
        "strategy_annualized_vol": float(backtest_df["strategy_return"].std() * np.sqrt(TRADING_DAYS_PER_YEAR)),
        "total_transaction_cost": float(backtest_df["transaction_cost"].sum()),
    }


In [ ]:
all_preds_sorted = all_preds.sort_index()
predicted_vol = np.exp(all_preds_sorted["y_pred_model_log"])
# The return realized *while holding* the position sized off predicted_vol
# for day t is the next trading day's return.
realized_next_return = feature_table_wf["log_return"].shift(-1).reindex(all_preds_sorted.index)

backtest_df = run_backtest(predicted_vol, realized_next_return,
                            target_daily_vol=0.01, max_leverage=2.0, cost_bps=7.5)
summary = backtest_summary(backtest_df)
for k, v in summary.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
fig, ax = plt.subplots()
ax.plot(backtest_df.index, backtest_df["strategy_equity"], label="Vol-targeting strategy")
ax.plot(backtest_df.index, backtest_df["buy_hold_equity"], label="Buy & hold")
ax.set_title("Backtest equity curve (net of transaction costs)")
ax.set_ylabel("Equity (starting = 1.0)")
ax.legend()
plt.show()


## 11. Explainability (SHAP)

TreeSHAP on the final model, checking whether sentiment features
meaningfully contribute alongside price autocorrelation -- not just
sanity-checking that the model learned volatility persistence and ignored
sentiment.

In [ ]:
"""TreeSHAP explainability for the LightGBM volatility model.

Fast and near-free for gradient boosting (unlike SHAP on deep nets). Shows
which features -- which sentiment lag, which price lag -- actually drive
predictions, useful both for the writeup and for sanity-checking that the
model isn't just learning volatility persistence and ignoring sentiment
entirely.
"""

import numpy as np
import pandas as pd
import shap


def compute_shap_values(model, X: pd.DataFrame):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X)
    return shap_values


def shap_feature_importance(shap_values, feature_names: list[str]) -> pd.DataFrame:
    mean_abs = np.abs(shap_values.values).mean(axis=0)
    return (pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True))


def sentiment_vs_price_contribution(importance_df: pd.DataFrame) -> dict:
    """Split total |SHAP| mass between sentiment-derived and price-derived
    features, to sanity-check the model isn't ignoring sentiment."""
    sentiment_mask = importance_df["feature"].str.startswith(
        ("sent_", "article_count", "pct_positive", "pct_negative", "has_news"))
    total = importance_df["mean_abs_shap"].sum()
    sentiment_share = importance_df.loc[sentiment_mask, "mean_abs_shap"].sum()
    return {
        "sentiment_share_pct": 100 * sentiment_share / total if total else 0.0,
        "price_share_pct": 100 * (total - sentiment_share) / total if total else 0.0,
    }


In [ ]:
X_explain = final_train[feature_cols].iloc[-500:]
shap_values = compute_shap_values(final_model, X_explain)
importance = shap_feature_importance(shap_values, feature_cols)
contribution = sentiment_vs_price_contribution(importance)

print(f"Sentiment features: {contribution['sentiment_share_pct']:.1f}% of total |SHAP|")
print(f"Price features:     {contribution['price_share_pct']:.1f}% of total |SHAP|")
display(importance.head(15))

shap.summary_plot(shap_values, X_explain, show=True)


## 12. Results summary

Re-run cells above with more Optuna trials, or a longer `PRICE_LOOKBACK_YEARS`
(which also widens the GDELT news backfill to match), for a stronger final
result -- the defaults above are tuned for a reasonable first end-to-end
Kaggle run.

In [ ]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"Walk-forward folds: {len(wf_results)}")
print(f"RMSE (log-vol)  model={overall['rmse_model']:.4f}  naive={overall['rmse_naive']:.4f}  garch={overall['rmse_garch']:.4f}")
print(f"MAE  (log-vol)  model={overall['mae_model']:.4f}  naive={overall['mae_naive']:.4f}  garch={overall['mae_garch']:.4f}")
print(f"Model vs naive:  {100*(overall['rmse_naive']-overall['rmse_model'])/overall['rmse_naive']:+.1f}% RMSE")
print(f"Model vs GARCH:  {100*(overall['rmse_garch']-overall['rmse_model'])/overall['rmse_garch']:+.1f}% RMSE")
print()
print(f"Backtest Sharpe   strategy={summary['strategy_sharpe']:.2f}  buy&hold={summary['buy_hold_sharpe']:.2f}")
print(f"Max drawdown      strategy={summary['strategy_max_drawdown']:.1%}  buy&hold={summary['buy_hold_max_drawdown']:.1%}")
print(f"Total return      strategy={summary['strategy_total_return']:.1%}  buy&hold={summary['buy_hold_total_return']:.1%}")
print()
print(f"SHAP: sentiment contributes {contribution['sentiment_share_pct']:.1f}% of total feature importance")
